In [ ]:
from pystac.client import Client
from odc.stac import load
import xarray as xr
from src.utils import mask_land, mask_deeps, make_indices, do_prediction, S2_BANDS
import joblib
from tqdm import tqdm

from ipyleaflet import basemaps
import folium

from dep_tools.grids import PACIFIC_GRID_10

# import warnings
# warnings.filterwarnings("ignore")

In [ ]:
catalog = Client.open("https://earth-search.aws.element84.com/v1")
collection = "sentinel-2-l2a"

In [ ]:
# Set location using defined study locations

# location = locations.suva
# bbox = location.bbox

# Set location using tile
tile_id = (130, 12)
geobox = PACIFIC_GRID_10.tile_geobox(tile_id)
bbox = list(geobox.geographic_extent.boundingbox)

# Date
daterange = "2024"

In [ ]:
items = catalog.search(
    collections=[collection],
    bbox=bbox,
    datetime=daterange,
    query={"eo:cloud_cover": {"lt": 50}},
).item_collection()

print(f"Found {len(items)} items")

In [ ]:
data = load(
    items,
    bbox=bbox,
    epsg="utm",
    measurements=S2_BANDS,
    chunks={"x": 2048, "y": 2048},
    nodata=0,
    groupby="solar_day"
)

# Mask clouds
mask = data.scl.isin([3, 8, 9, 10])
data = data.where(~mask)
data = make_indices(data)

# Mask land
data = mask_land(data)

# # Mask deep water
data = mask_deeps(data)

data = data.drop_vars("scl")

data

In [ ]:
model = joblib.load("models/2025_06_13_rf.joblib")

predictions_list = []

for day in tqdm(data.time):
    data_day = data.sel(time=day).compute()
    predictions_list.append(do_prediction(data_day, model))

# Concatenate them all together again
predictions = xr.concat(predictions_list, dim="time").to_dataset(name="elevation")

predictions

In [ ]:
# Plot all the timesteps
predictions.elevation.plot.imshow(col="time", col_wrap=2, cmap="Blues_r", robust=True, size=6)

In [ ]:
# Clean up the data by removing pixels that only had predictions sometimes
count = predictions.elevation.count(dim="time")
total = len(predictions.time)

mask = count > (total * 0.15)  # At least X% of the time there was a prediction

mean = predictions.elevation.mean(dim="time")
stdev = predictions.elevation.std(dim="time")

stdev_mask = stdev < 3

mean_countmasked = mean.where(mask)
mean_stdevmasked = mean.where(stdev_mask)

# Make a fancy map
centroid = list(predictions.odc.geobox.geographic_extent.centroid.coords[0])[::-1]
m = folium.Map(location=centroid, zoom_start=12)
_ = folium.TileLayer(tiles=basemaps.Esri.WorldImagery).add_to(m)

count.odc.add_to(m, cmap="Reds", name="Count")
stdev.odc.add_to(m, cmap="Reds", name="Stdev")

mean.odc.add_to(m, cmap="Blues_r", name="Depth")

mean_stdevmasked.odc.add_to(m, cmap="Blues_r", name="Depth (stdev masked)")
mean_countmasked.odc.add_to(m, cmap="Blues_r", name="Depth (count masked)")

folium.LayerControl().add_to(m)

m

In [ ]:
count.plot.imshow()